# Import Data

In [1]:
import pandas as pd
import numpy as np

In [2]:
import re

def extract_drive_id(url):
    """
    Hàm nhận vào URL Google Drive và trả về File ID
    Hỗ trợ cả dạng /file/d/ và open?id=
    """
    patterns = [
        r"https://drive\.google\.com/file/d/([a-zA-Z0-9_-]+)",  # dạng /file/d/ID
        r"id=([a-zA-Z0-9_-]+)"                                  # dạng ?id=ID
    ]

    for pattern in patterns:
        match = re.search(pattern, url)
        if match:
            return match.group(1)
    return None

# Sử dụng
url = "https://drive.google.com/open?id=1JssLn9uCE5WixaLP0WYTr3Satcjxf_f0&usp=drive_copy"
file_id = extract_drive_id(url)

if file_id:
    print("🎯 File ID extracted:", file_id)
else:
    print("❌ Không tìm thấy file ID từ link.")

# Bạn có thể dùng luôn với gdown:
import gdown
gdown.download(f"https://drive.google.com/uc?id={file_id}", "du_lieu_vong_2.xlsx", quiet=False)


🎯 File ID extracted: 1JssLn9uCE5WixaLP0WYTr3Satcjxf_f0


Downloading...
From: https://drive.google.com/uc?id=1JssLn9uCE5WixaLP0WYTr3Satcjxf_f0
To: /content/du_lieu_vong_2.xlsx
100%|██████████| 2.11M/2.11M [00:00<00:00, 30.4MB/s]


'du_lieu_vong_2.xlsx'

In [3]:
import pandas as pd

# Đường dẫn đến file (nếu đã tải về Colab)
file_path = "du_lieu_vong_2.xlsx"

# Đọc từng sheet
balance_sheet = pd.read_excel(file_path, sheet_name='1. Balance Sheet')
income_statement = pd.read_excel(file_path, sheet_name='2. Income Statement')
cash_flow = pd.read_excel(file_path, sheet_name='3. Cash Flow')

In [4]:
balance_sheet.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8010 entries, 0 to 8009
Data columns (total 19 columns):
 #   Column                                 Non-Null Count  Dtype 
---  ------                                 --------------  ----- 
 0   Công ty                                8010 non-null   int64 
 1   Năm                                    8010 non-null   int64 
 2   A. TÀI SẢN NGẮN HẠN                    8010 non-null   object
 3   I. Tiền và các khoản tương đương tiền  8010 non-null   object
 4   II. Đầu tư tài chính ngắn hạn          8010 non-null   object
 5   III. Các khoản phải thu ngắn hạn       8010 non-null   object
 6   IV. Hàng tồn kho                       8010 non-null   object
 7   V. Tài sản ngắn hạn khác               8010 non-null   object
 8   B. TÀI SẢN DÀI HẠN                     8010 non-null   object
 9   I. Các khoản phải thu dài hạn          8010 non-null   object
 10  II. Tài sản cố định                    8010 non-null   object
 11  III. Bất động sản

In [5]:
income_statement.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8010 entries, 0 to 8009
Data columns (total 18 columns):
 #   Column                                         Non-Null Count  Dtype  
---  ------                                         --------------  -----  
 0   Công ty                                        8010 non-null   int64  
 1   Năm                                            8010 non-null   int64  
 2   Doanh thu bán hàng & cung cấp dịch vụ          8002 non-null   float64
 3   Các khoản giảm trừ doanh thu                   8002 non-null   float64
 4   Doanh thu thuần bán hàng và cung cấp dịch vụ   8002 non-null   float64
 5   Giá vốn hàng bán                               8002 non-null   float64
 6   Lợi nhuận gộp về bán hàng và cung cấp dịch vụ  8002 non-null   float64
 7   Doanh thu hoạt động tài chính                  8003 non-null   float64
 8   Chi phí tài chính                              8003 non-null   float64
 9   - Trong đó: Chi phí lãi vay                    8004 

In [6]:
cash_flow.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8010 entries, 0 to 8009
Data columns (total 6 columns):
 #   Column                                         Non-Null Count  Dtype  
---  ------                                         --------------  -----  
 0   Công ty                                        8010 non-null   int64  
 1   Năm                                            8010 non-null   int64  
 2   Lưu Chuyển Tiền Thuần Từ Hoạt Động Kinh Doanh  7718 non-null   float64
 3   Lưu Chuyển Tiền Thuần Từ Hoạt Động Đầu Tư      7718 non-null   float64
 4   Lưu Chuyển Tiền Thuần Từ Hoạt Động Tài Chính   7718 non-null   float64
 5   Lưu Chuyển Tiền Thuần Trong Kỳ                 7718 non-null   float64
dtypes: float64(4), int64(2)
memory usage: 375.6 KB


# Clean data

## Balance Sheet

### Thay '-' bằng NaN để xử lý số

In [7]:
# Thay '-' bằng NaN để xử lý số
balance_sheet.replace('-', np.nan, inplace=True)

### Chuyển tất cả các cột (trừ 'cong_ty' và 'nam') sang số

In [8]:
# Chuyển tất cả các cột (trừ 'cong_ty' và 'nam') sang số
cols_to_convert = balance_sheet.columns.difference(['cong_ty', 'nam'])
balance_sheet[cols_to_convert] = balance_sheet[cols_to_convert].apply(pd.to_numeric, errors='coerce')


### Kiểm tra xem có dòng hay cột nào NaN hết không ? ==> Không có

In [9]:
# Đếm số dòng toàn bộ là NaN (sau khi đã loại trừ các cột 'cong_ty' và 'nam' là metadata)
rows_all_nan = balance_sheet.drop(columns=['cong_ty', 'nam'], errors='ignore').isna().all(axis=1).sum()

# Đếm số cột toàn bộ là NaN
cols_all_nan = balance_sheet.isna().all(axis=0).sum()

(rows_all_nan, cols_all_nan)

(np.int64(0), np.int64(0))

### Clean theo logic kế toán

#### Clean theo A + B = C + D

In [10]:
# Tạo bản sao để điền các cột còn lại
balance_sheet_filled_all = balance_sheet.copy()

In [11]:
import pandas as pd
import numpy as np

def fill_balance_sheet(df):
    df_filled = df.copy()
    cols = ['A. TÀI SẢN NGẮN HẠN', 'B. TÀI SẢN DÀI HẠN', 'C. NỢ PHẢI TRẢ', 'D. VỐN CHỦ SỞ HỮU']

    # Bước 1: Gán giá trị = 0 nếu công thức đúng và thiếu <= 2 giá trị
    for idx, row in df_filled.iterrows():
        a, b, c, d = row[cols[0]], row[cols[1]], row[cols[2]], row[cols[3]]
        left = (a if pd.notna(a) else 0) + (b if pd.notna(b) else 0)
        right = (c if pd.notna(c) else 0) + (d if pd.notna(d) else 0)
        nan_flags = [pd.isna(x) for x in [a, b, c, d]]
        nan_count = sum(nan_flags)

        if np.isclose(left, right, atol=1e-2) and nan_count <= 2:
            if pd.isna(a): df_filled.at[idx, cols[0]] = 0
            if pd.isna(b): df_filled.at[idx, cols[1]] = 0
            if pd.isna(c): df_filled.at[idx, cols[2]] = 0
            if pd.isna(d): df_filled.at[idx, cols[3]] = 0

    # Bước 2: Tính phần còn lại dựa vào công thức
    # Fill A
    cond_a = (
        df_filled[cols[0]].isna() &
        df_filled[cols[1]].notna() &
        df_filled[cols[2]].notna() &
        df_filled[cols[3]].notna()
    )
    df_filled.loc[cond_a, cols[0]] = (
        df_filled[cols[2]] + df_filled[cols[3]] - df_filled[cols[1]]
    )

    # Fill B
    cond_b = (
        df_filled[cols[1]].isna() &
        df_filled[cols[0]].notna() &
        df_filled[cols[2]].notna() &
        df_filled[cols[3]].notna()
    )
    df_filled.loc[cond_b, cols[1]] = (
        df_filled[cols[2]] + df_filled[cols[3]] - df_filled[cols[0]]
    )

    # Fill C
    cond_c = (
        df_filled[cols[2]].isna() &
        df_filled[cols[0]].notna() &
        df_filled[cols[1]].notna() &
        df_filled[cols[3]].notna()
    )
    df_filled.loc[cond_c, cols[2]] = (
        df_filled[cols[0]] + df_filled[cols[1]] - df_filled[cols[3]]
    )

    # Fill D
    cond_d = (
        df_filled[cols[3]].isna() &
        df_filled[cols[0]].notna() &
        df_filled[cols[1]].notna() &
        df_filled[cols[2]].notna()
    )
    df_filled.loc[cond_d, cols[3]] = (
        df_filled[cols[0]] + df_filled[cols[1]] - df_filled[cols[2]]
    )

    return df_filled

balance_sheet_filled_all = fill_balance_sheet(balance_sheet_filled_all)

#### **Clean theo**
- A = I + II + ... + V
- B = I + II + ... + VI
- C = I + II
- D

In [12]:
# --- A. TÀI SẢN NGẮN HẠN ---
component_a_cols = [
    'I. Tiền và các khoản tương đương tiền',
    'II. Đầu tư tài chính ngắn hạn',
    'III. Các khoản phải thu ngắn hạn',
    'IV. Hàng tồn kho',
    'V. Tài sản ngắn hạn khác'
]

# Fill A nếu có đủ thành phần
cond_fill_a = (
    balance_sheet_filled_all['A. TÀI SẢN NGẮN HẠN'].isna() &
    balance_sheet_filled_all[component_a_cols].notna().all(axis=1)
)
balance_sheet_filled_all.loc[cond_fill_a, 'A. TÀI SẢN NGẮN HẠN'] = balance_sheet_filled_all[component_a_cols].sum(axis=1)

# Fill thành phần nếu A có và còn thiếu đúng 1 thành phần
for col in component_a_cols:
    others = [c for c in component_a_cols if c != col]
    cond_missing_col = (
        balance_sheet_filled_all[col].isna() &
        balance_sheet_filled_all['A. TÀI SẢN NGẮN HẠN'].notna() &
        balance_sheet_filled_all[others].notna().all(axis=1)
    )
    value = balance_sheet_filled_all['A. TÀI SẢN NGẮN HẠN'] - balance_sheet_filled_all[others].sum(axis=1)
    balance_sheet_filled_all.loc[cond_missing_col, col] = value

# Nếu A và >= 1 thành phần bị thiếu, nhưng tổng các thành phần còn lại đã bằng A => gán phần thiếu = 0
for idx, row in balance_sheet_filled_all.iterrows():
    if pd.notna(row['A. TÀI SẢN NGẮN HẠN']):
        values = row[component_a_cols]
        missing = values.isna()
        if 0 < missing.sum() <= 4:  # chỉ xử lý khi thiếu 1–2 giá trị
            if np.isclose(values[~missing].sum(), row['A. TÀI SẢN NGẮN HẠN'], atol=1e-2):
                for col in values.index[missing]:
                    balance_sheet_filled_all.at[idx, col] = 0

# --- B. TÀI SẢN DÀI HẠN ---
component_b_cols = [
    'I. Các khoản phải thu dài hạn',
    'II. Tài sản cố định',
    'III. Bất động sản đầu tư',
    'IV. Tài sản dở dang dài hạn',
    'V. Đầu tư tài chính dài hạn',
    'VI. Tài sản dài hạn khác'
]

cond_fill_b = (
    balance_sheet_filled_all['B. TÀI SẢN DÀI HẠN'].isna() &
    balance_sheet_filled_all[component_b_cols].notna().all(axis=1)
)
balance_sheet_filled_all.loc[cond_fill_b, 'B. TÀI SẢN DÀI HẠN'] = balance_sheet_filled_all[component_b_cols].sum(axis=1)

for col in component_b_cols:
    others = [c for c in component_b_cols if c != col]
    cond_fill_component = (
        balance_sheet_filled_all[col].isna() &
        balance_sheet_filled_all['B. TÀI SẢN DÀI HẠN'].notna() &
        balance_sheet_filled_all[others].notna().all(axis=1)
    )
    value = balance_sheet_filled_all['B. TÀI SẢN DÀI HẠN'] - balance_sheet_filled_all[others].sum(axis=1)
    balance_sheet_filled_all.loc[cond_fill_component, col] = value

# Nếu B có và tổng các thành phần còn lại = B → fill phần thiếu = 0
for idx, row in balance_sheet_filled_all.iterrows():
    if pd.notna(row['B. TÀI SẢN DÀI HẠN']):
        values = row[component_b_cols]
        missing = values.isna()
        if 0 < missing.sum() <= 5:
            if np.isclose(values[~missing].sum(), row['B. TÀI SẢN DÀI HẠN'], atol=1e-2):
                for col in values.index[missing]:
                    balance_sheet_filled_all.at[idx, col] = 0


# --- C. NỢ PHẢI TRẢ ---
cond_fill_c = (
    balance_sheet_filled_all['C. NỢ PHẢI TRẢ'].isna() &
    balance_sheet_filled_all['I. Nợ ngắn hạn'].notna() &
    balance_sheet_filled_all['II. Nợ dài hạn'].notna()
)
balance_sheet_filled_all.loc[cond_fill_c, 'C. NỢ PHẢI TRẢ'] = (
    balance_sheet_filled_all['I. Nợ ngắn hạn'] + balance_sheet_filled_all['II. Nợ dài hạn']
)

cond_fill_i = (
    balance_sheet_filled_all['I. Nợ ngắn hạn'].isna() &
    balance_sheet_filled_all['C. NỢ PHẢI TRẢ'].notna() &
    balance_sheet_filled_all['II. Nợ dài hạn'].notna()
)
balance_sheet_filled_all.loc[cond_fill_i, 'I. Nợ ngắn hạn'] = (
    balance_sheet_filled_all['C. NỢ PHẢI TRẢ'] - balance_sheet_filled_all['II. Nợ dài hạn']
)

cond_fill_ii = (
    balance_sheet_filled_all['II. Nợ dài hạn'].isna() &
    balance_sheet_filled_all['C. NỢ PHẢI TRẢ'].notna() &
    balance_sheet_filled_all['I. Nợ ngắn hạn'].notna()
)
balance_sheet_filled_all.loc[cond_fill_ii, 'II. Nợ dài hạn'] = (
    balance_sheet_filled_all['C. NỢ PHẢI TRẢ'] - balance_sheet_filled_all['I. Nợ ngắn hạn']
)

# Nếu C đã có và thiếu 1 thành phần mà tổng phần còn lại = C → fill phần thiếu = 0
for idx, row in balance_sheet_filled_all.iterrows():
    if pd.notna(row['C. NỢ PHẢI TRẢ']):
        short = row['I. Nợ ngắn hạn']
        long_ = row['II. Nợ dài hạn']
        if pd.isna(short) and pd.notna(long_) and np.isclose(long_, row['C. NỢ PHẢI TRẢ'], atol=1e-2):
            balance_sheet_filled_all.at[idx, 'I. Nợ ngắn hạn'] = 0
        elif pd.isna(long_) and pd.notna(short) and np.isclose(short, row['C. NỢ PHẢI TRẢ'], atol=1e-2):
            balance_sheet_filled_all.at[idx, 'II. Nợ dài hạn'] = 0


In [13]:
balance_sheet_filled_all["sai_số_tài_sản_vs_nguồn_vốn"] = (
    (balance_sheet_filled_all["A. TÀI SẢN NGẮN HẠN"] + balance_sheet_filled_all["B. TÀI SẢN DÀI HẠN"]) -
    (balance_sheet_filled_all["C. NỢ PHẢI TRẢ"] + balance_sheet_filled_all["D. VỐN CHỦ SỞ HỮU"])
)


In [14]:
# Lọc các dòng có ít nhất một trong bốn cột A, B, C, D bị thiếu (NaN)
missing_abc_d = balance_sheet_filled_all[
    balance_sheet_filled_all[[
        "A. TÀI SẢN NGẮN HẠN",
        "B. TÀI SẢN DÀI HẠN",
        "C. NỢ PHẢI TRẢ",
        "D. VỐN CHỦ SỞ HỮU"
    ]].isna().any(axis=1)
]

In [15]:
missing_abc_d

,Công ty,Năm,A. TÀI SẢN NGẮN HẠN,I. Tiền và các khoản tương đương tiền,II. Đầu tư tài chính ngắn hạn,III. Các khoản phải thu ngắn hạn,IV. Hàng tồn kho,V. Tài sản ngắn hạn khác,B. TÀI SẢN DÀI HẠN,I. Các khoản phải thu dài hạn,II. Tài sản cố định,III. Bất động sản đầu tư,IV. Tài sản dở dang dài hạn,V. Đầu tư tài chính dài hạn,VI. Tài sản dài hạn khác,C. NỢ PHẢI TRẢ,I. Nợ ngắn hạn,II. Nợ dài hạn,D. VỐN CHỦ SỞ HỮU,sai_số_tài_sản_vs_nguồn_vốn
230,47,2019,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1880,377,2019,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2920,585,2019,256828526.0,238810830.0,0.0,0.0,0.0,18017696.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,257163981.0,NaN
2921,585,2020,246594561.0,227553468.0,0.0,0.0,0.0,19041093.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,246930016.0,NaN
4423,885,2022,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Final Data

In [16]:
# Đếm số ô thiếu trước và sau khi fill
missing_before = balance_sheet.isna().sum()
missing_after = balance_sheet_filled_all.isna().sum()

# Kết hợp thành bảng so sánh
missing_comparison = pd.DataFrame({
    'Thiếu trước fill': missing_before,
    'Thiếu sau fill': missing_after,
    'Số ô được fill': missing_before - missing_after
})

missing_comparison


,Thiếu trước fill,Thiếu sau fill,Số ô được fill
A. TÀI SẢN NGẮN HẠN,3.0,3,0.0
B. TÀI SẢN DÀI HẠN,2318.0,5,2313.0
C. NỢ PHẢI TRẢ,755.0,5,750.0
Công ty,0.0,0,0.0
D. VỐN CHỦ SỞ HỮU,16.0,3,13.0
I. Các khoản phải thu dài hạn,6653.0,2333,4320.0
I. Nợ ngắn hạn,886.0,776,110.0
I. Tiền và các khoản tương đương tiền,23.0,5,18.0
II. Nợ dài hạn,6414.0,776,5638.0
II. Tài sản cố định,3543.0,2320,1223.0


## Income Statement

### Chuyển toàn bộ cột tài chính về dạng số (float)

In [17]:
cols_to_convert = [col for col in income_statement.columns if col not in ['Công ty', 'Năm']]
income_statement[cols_to_convert] = income_statement[cols_to_convert].apply(pd.to_numeric, errors='coerce')


In [18]:
income_statement.isna().sum()

,0
Công ty,0
Năm,0
Doanh thu bán hàng & cung cấp dịch vụ,8
Các khoản giảm trừ doanh thu,8
Doanh thu thuần bán hàng và cung cấp dịch vụ,8
Giá vốn hàng bán,8
Lợi nhuận gộp về bán hàng và cung cấp dịch vụ,8
Doanh thu hoạt động tài chính,7
Chi phí tài chính,7
- Trong đó: Chi phí lãi vay,6


### Clean theo logic kế toán

- Doanh thu thuần	= Doanh thu bán hàng – Các khoản giảm trừ
- Lợi nhuận gộp	= Doanh thu thuần – Giá vốn hàng bán
- Lợi nhuận thuần từ HĐKD	= Lợi nhuận gộp + Doanh thu tài chính – Chi phí tài chính – Chi phí bán hàng – Chi phí QLDN
- Lợi nhuận kế toán trước thuế	= Lợi nhuận thuần từ HĐKD + Thu nhập khác – Chi phí khác
- Lợi nhuận/(lỗ) khác	= Thu nhập khác – Chi phí khác
- Chi phí tài chính	= Chi phí lãi vay + Chi phí tài chính khác (nếu tách riêng)

In [20]:
# Copy để xử lý
is_filled = income_statement.copy()

#### 1. Doanh thu thuần = Doanh thu bán hàng – Khoản giảm trừ

In [23]:
# 1. Doanh thu thuần = Doanh thu bán hàng – Khoản giảm trừ
cond = is_filled['Doanh thu thuần bán hàng và cung cấp dịch vụ'].isna() & \
       is_filled['Doanh thu bán hàng & cung cấp dịch vụ'].notna() & \
       is_filled['Các khoản giảm trừ doanh thu'].notna()
is_filled.loc[cond, 'Doanh thu thuần bán hàng và cung cấp dịch vụ'] = (
    is_filled['Doanh thu bán hàng & cung cấp dịch vụ'] + is_filled['Các khoản giảm trừ doanh thu']
)

# Ngược lại
cond_doanhthu = is_filled['Doanh thu bán hàng & cung cấp dịch vụ'].isna() & \
                is_filled['Doanh thu thuần bán hàng và cung cấp dịch vụ'].notna() & \
                is_filled['Các khoản giảm trừ doanh thu'].notna()
is_filled.loc[cond_doanhthu, 'Doanh thu bán hàng & cung cấp dịch vụ'] = (
    is_filled['Doanh thu thuần bán hàng và cung cấp dịch vụ'] - is_filled['Các khoản giảm trừ doanh thu']
)

cond_giamtru = is_filled['Các khoản giảm trừ doanh thu'].isna() & \
               is_filled['Doanh thu thuần bán hàng và cung cấp dịch vụ'].notna() & \
               is_filled['Doanh thu bán hàng & cung cấp dịch vụ'].notna()
is_filled.loc[cond_giamtru, 'Các khoản giảm trừ doanh thu'] = (
    - is_filled['Doanh thu bán hàng & cung cấp dịch vụ'] + is_filled['Doanh thu thuần bán hàng và cung cấp dịch vụ']
)

#### 2. Lợi nhuận gộp = Doanh thu thuần – Giá vốn hàng bán

In [24]:
# 2. Lợi nhuận gộp = Doanh thu thuần – Giá vốn hàng bán
cond = is_filled['Lợi nhuận gộp về bán hàng và cung cấp dịch vụ'].isna() & \
       is_filled['Doanh thu thuần bán hàng và cung cấp dịch vụ'].notna() & \
       is_filled['Giá vốn hàng bán'].notna()
is_filled.loc[cond, 'Lợi nhuận gộp về bán hàng và cung cấp dịch vụ'] = (
    is_filled['Doanh thu thuần bán hàng và cung cấp dịch vụ'] - is_filled['Giá vốn hàng bán']
)

# Ngược lại
cond_doanhthuthuan = is_filled['Doanh thu thuần bán hàng và cung cấp dịch vụ'].isna() & \
                     is_filled['Lợi nhuận gộp về bán hàng và cung cấp dịch vụ'].notna() & \
                     is_filled['Giá vốn hàng bán'].notna()
is_filled.loc[cond_doanhthuthuan, 'Doanh thu thuần bán hàng và cung cấp dịch vụ'] = (
    is_filled['Lợi nhuận gộp về bán hàng và cung cấp dịch vụ'] + is_filled['Giá vốn hàng bán']
)

cond_giavon = is_filled['Giá vốn hàng bán'].isna() & \
              is_filled['Doanh thu thuần bán hàng và cung cấp dịch vụ'].notna() & \
              is_filled['Lợi nhuận gộp về bán hàng và cung cấp dịch vụ'].notna()
is_filled.loc[cond_giavon, 'Giá vốn hàng bán'] = (
    is_filled['Doanh thu thuần bán hàng và cung cấp dịch vụ'] - is_filled['Lợi nhuận gộp về bán hàng và cung cấp dịch vụ']
)

#### 3. Lợi nhuận thuần từ hoạt động kinh doanh = Lợi nhuận gộp + Doanh thu tài chính + Lãi/Lỗ  trong c.ty liên kết, liên doanh – Chi phí tài chính – Chi phí bán hàng – Chi phí quản lý


In [26]:
import numpy as np

# Đặt tên cột cho dễ code
col_LNT_HDKD = 'Lợi nhuận thuần từ hoạt động kinh doanh'
col_LN_Gop = 'Lợi nhuận gộp về bán hàng và cung cấp dịch vụ'
col_DT_TC = 'Doanh thu hoạt động tài chính'
col_Lai_LienKet = 'Phần lãi (lỗ) trong c.ty liên kết, liên doanh'
col_CP_TC = 'Chi phí tài chính'
col_CP_BanHang = 'Chi phí bán hàng'
col_CP_QLDN = 'Chi phí quản lý doanh nghiệp'

cols = [col_LNT_HDKD, col_LN_Gop, col_DT_TC, col_Lai_LienKet, col_CP_TC, col_CP_BanHang, col_CP_QLDN]

# 1. Nếu chỉ thiếu đúng 1 thành phần, tính ngược lại từ công thức
for idx, row in is_filled.iterrows():
    values = row[cols]
    na_count = values.isna().sum()
    if na_count == 1:
        na_col = values.index[values.isna()][0]
        known_sum = values.drop(na_col).sum()
        if na_col == col_LNT_HDKD:
            is_filled.at[idx, col_LNT_HDKD] = known_sum
        else:
            is_filled.at[idx, na_col] = row[col_LNT_HDKD] - values.drop(na_col).sum()

# 2. Nếu 2 vế đã bằng nhau (hoặc đủ số lượng thành phần để kiểm tra), fill các thành phần còn lại = 0
for idx, row in is_filled.iterrows():
    values = row[cols]
    na_cols = values.index[values.isna()]
    # Kiểm tra vế trái và phải đã bằng nhau chưa (tức là LNT_HDKD = tổng các thành phần)
    rhs = row[col_LN_Gop] + row[col_DT_TC] + row[col_Lai_LienKet] + row[col_CP_TC] + row[col_CP_BanHang] + row[col_CP_QLDN]
    lhs = row[col_LNT_HDKD]
    if np.isclose(lhs, rhs, atol=1e-3):  # Có thể chỉnh atol cho phù hợp dữ liệu thực tế
        # Fill các thành phần còn lại = 0
        for na_col in na_cols:
            if na_col != col_LNT_HDKD:  # Không fill vào cột kết quả
                is_filled.at[idx, na_col] = 0


#### Lợi nhuận trước thuế = Lợi nhuận HĐKD + Thu nhập khác – Chi phí khác


In [27]:
import numpy as np

cols = [
    'Tổng lợi nhuận kế toán trước thuế',
    'Lợi nhuận thuần từ hoạt động kinh doanh',
    'Thu nhập khác',
    'Chi phí khác'
]

# 1. Nếu chỉ thiếu đúng 1 thành phần thì tính ngược lại
for idx, row in is_filled.iterrows():
    vals = row[cols]
    na_count = vals.isna().sum()
    if na_count == 1:
        na_col = vals.index[vals.isna()][0]
        if na_col == 'Tổng lợi nhuận kế toán trước thuế':
            is_filled.at[idx, 'Tổng lợi nhuận kế toán trước thuế'] = (
                row['Lợi nhuận thuần từ hoạt động kinh doanh'] +
                row['Thu nhập khác'] -
                row['Chi phí khác']
            )
        elif na_col == 'Lợi nhuận thuần từ hoạt động kinh doanh':
            is_filled.at[idx, 'Lợi nhuận thuần từ hoạt động kinh doanh'] = (
                row['Tổng lợi nhuận kế toán trước thuế'] -
                row['Thu nhập khác'] +
                row['Chi phí khác']
            )
        elif na_col == 'Thu nhập khác':
            is_filled.at[idx, 'Thu nhập khác'] = (
                row['Tổng lợi nhuận kế toán trước thuế'] -
                row['Lợi nhuận thuần từ hoạt động kinh doanh'] +
                row['Chi phí khác']
            )
        elif na_col == 'Chi phí khác':
            is_filled.at[idx, 'Chi phí khác'] = (
                row['Lợi nhuận thuần từ hoạt động kinh doanh'] +
                row['Thu nhập khác'] -
                row['Tổng lợi nhuận kế toán trước thuế']
            )

# 2. Nếu 2 vế đã bằng nhau, fill các thành phần còn lại = 0 (trừ cột kết quả)
for idx, row in is_filled.iterrows():
    vals = row[cols]
    na_cols = vals.index[vals.isna()]
    # Vế trái = vế phải
    lhs = row['Tổng lợi nhuận kế toán trước thuế']
    rhs = row['Lợi nhuận thuần từ hoạt động kinh doanh'] + row['Thu nhập khác'] - row['Chi phí khác']
    # Nếu đã đủ 2 thành phần hoặc đủ điều kiện tính, kiểm tra equality
    if pd.notna(lhs) and pd.notna(row['Lợi nhuận thuần từ hoạt động kinh doanh']) and \
       pd.notna(row['Thu nhập khác']) and pd.notna(row['Chi phí khác']):
        if np.isclose(lhs, rhs, atol=1e-3):
            for na_col in na_cols:
                if na_col != 'Tổng lợi nhuận kế toán trước thuế':  # Không fill vào cột kết quả
                    is_filled.at[idx, na_col] = 0


#### Lợi nhuận/(lỗ) khác	= Thu nhập khác – Chi phí khác



In [28]:
# 5. Other profit = Other income - Other expenses
cond_lai_lo_khac = is_filled['Lợi nhuận/(lỗ) khác'].isna() & \
    is_filled['Thu nhập khác'].notna() & \
    is_filled['Chi phí khác'].notna()
is_filled.loc[cond_lai_lo_khac, 'Lợi nhuận/(lỗ) khác'] = (
    is_filled['Thu nhập khác'] - is_filled['Chi phí khác']
)

# Reverse logic for other income or other expense
cond_thu_nhap_khac = is_filled['Thu nhập khác'].isna() & \
    is_filled['Lợi nhuận/(lỗ) khác'].notna() & \
    is_filled['Chi phí khác'].notna()
is_filled.loc[cond_thu_nhap_khac, 'Thu nhập khác'] = (
    is_filled['Lợi nhuận/(lỗ) khác'] + is_filled['Chi phí khác']
)

cond_chi_phi_khac = is_filled['Chi phí khác'].isna() & \
    is_filled['Thu nhập khác'].notna() & \
    is_filled['Lợi nhuận/(lỗ) khác'].notna()
is_filled.loc[cond_chi_phi_khac, 'Chi phí khác'] = (
    is_filled['Thu nhập khác'] - is_filled['Lợi nhuận/(lỗ) khác']
)

In [29]:
# Đặt tên cột để dễ đọc và tránh lỗi chính tả
col_doanh_thu = 'Doanh thu bán hàng & cung cấp dịch vụ'
col_loi_nhuan_thuan = 'Lợi nhuận thuần từ hoạt động kinh doanh'

# Kiểm tra điều kiện
condition = (
    is_filled[col_doanh_thu].notna() &
    is_filled[col_loi_nhuan_thuan].notna() &
    (is_filled[col_doanh_thu] < is_filled[col_loi_nhuan_thuan])
)

# Đếm số dòng thỏa mãn
num_invalid_rows = condition.sum()

print(f"Số dòng có {col_doanh_thu} < {col_loi_nhuan_thuan}: {num_invalid_rows}")


Số dòng có Doanh thu bán hàng & cung cấp dịch vụ < Lợi nhuận thuần từ hoạt động kinh doanh: 61


### Final Data

In [30]:
# Đếm số ô thiếu trước và sau khi fill
missing_before = income_statement.isna().sum()
missing_after = is_filled.isna().sum()

# Kết hợp thành bảng so sánh
missing_comparison = pd.DataFrame({
    'Thiếu trước fill': missing_before,
    'Thiếu sau fill': missing_after,
    'Số ô được fill': missing_before - missing_after
})

missing_comparison


,Thiếu trước fill,Thiếu sau fill,Số ô được fill
Công ty,0,0,0
Năm,0,0,0
Doanh thu bán hàng & cung cấp dịch vụ,8,8,0
Các khoản giảm trừ doanh thu,8,8,0
Doanh thu thuần bán hàng và cung cấp dịch vụ,8,8,0
Giá vốn hàng bán,8,8,0
Lợi nhuận gộp về bán hàng và cung cấp dịch vụ,8,7,1
Doanh thu hoạt động tài chính,7,7,0
Chi phí tài chính,7,7,0
- Trong đó: Chi phí lãi vay,6,6,0


## Cash Flow

### Chuyển toàn bộ cột tài chính về dạng số

In [34]:
# Chuyển tất cả cột tài chính về dạng số
cols_to_convert = [col for col in cash_flow.columns if col not in ['Công ty', 'Năm']]
cash_flow[cols_to_convert] = cash_flow[cols_to_convert].apply(pd.to_numeric, errors='coerce')


### Clean theo logic kế toán

In [35]:
# Tạo bản sao để xử lý
cf_filled = cash_flow.copy()

In [36]:
# Logic: Lưu chuyển tiền thuần trong kỳ = HĐKD + HĐ đầu tư + HĐ tài chính
cond_fill_cf_total = (
    cf_filled['Lưu Chuyển Tiền Thuần Trong Kỳ'].isna() &
    cf_filled['Lưu Chuyển Tiền Thuần Từ Hoạt Động Kinh Doanh'].notna() &
    cf_filled['Lưu Chuyển Tiền Thuần Từ Hoạt Động Đầu Tư'].notna() &
    cf_filled['Lưu Chuyển Tiền Thuần Từ Hoạt Động Tài Chính'].notna()
)

cf_filled.loc[cond_fill_cf_total, 'Lưu Chuyển Tiền Thuần Trong Kỳ'] = (
    cf_filled['Lưu Chuyển Tiền Thuần Từ Hoạt Động Kinh Doanh'] +
    cf_filled['Lưu Chuyển Tiền Thuần Từ Hoạt Động Đầu Tư'] +
    cf_filled['Lưu Chuyển Tiền Thuần Từ Hoạt Động Tài Chính']
)

# Ngược lại: nếu thiếu từng phần trong 3 thành phần và có tổng + 2 phần còn lại
components = [
    'Lưu Chuyển Tiền Thuần Từ Hoạt Động Kinh Doanh',
    'Lưu Chuyển Tiền Thuần Từ Hoạt Động Đầu Tư',
    'Lưu Chuyển Tiền Thuần Từ Hoạt Động Tài Chính'
]

for comp in components:
    others = [c for c in components if c != comp]
    cond = (
        cf_filled[comp].isna() &
        cf_filled['Lưu Chuyển Tiền Thuần Trong Kỳ'].notna() &
        cf_filled[others].notna().all(axis=1)
    )
    cf_filled.loc[cond, comp] = (
        cf_filled['Lưu Chuyển Tiền Thuần Trong Kỳ'] - cf_filled[others].sum(axis=1)
    )


### Final Data

In [37]:
# Đếm số ô thiếu trước và sau khi fill
missing_before = cash_flow.isna().sum()
missing_after = cf_filled.isna().sum()

# Kết hợp thành bảng so sánh
missing_comparison = pd.DataFrame({
    'Thiếu trước fill': missing_before,
    'Thiếu sau fill': missing_after,
    'Số ô được fill': missing_before - missing_after
})
missing_comparison

,Thiếu trước fill,Thiếu sau fill,Số ô được fill
Công ty,0,0,0
Năm,0,0,0
Lưu Chuyển Tiền Thuần Từ Hoạt Động Kinh Doanh,292,292,0
Lưu Chuyển Tiền Thuần Từ Hoạt Động Đầu Tư,292,292,0
Lưu Chuyển Tiền Thuần Từ Hoạt Động Tài Chính,292,292,0
Lưu Chuyển Tiền Thuần Trong Kỳ,292,292,0


In [38]:
# Lọc các dòng có ít nhất một trong bốn cột A, B, C, D bị thiếu (NaN)
missing_abc_d = cf_filled[
    cf_filled[[
        "Lưu Chuyển Tiền Thuần Từ Hoạt Động Kinh Doanh",
        "Lưu Chuyển Tiền Thuần Từ Hoạt Động Đầu Tư",
        "Lưu Chuyển Tiền Thuần Từ Hoạt Động Tài Chính",
        "Lưu Chuyển Tiền Thuần Trong Kỳ"
    ]].isna().any(axis=1)
]

In [39]:
missing_abc_d

,Công ty,Năm,Lưu Chuyển Tiền Thuần Từ Hoạt Động Kinh Doanh,Lưu Chuyển Tiền Thuần Từ Hoạt Động Đầu Tư,Lưu Chuyển Tiền Thuần Từ Hoạt Động Tài Chính,Lưu Chuyển Tiền Thuần Trong Kỳ
0,1,2019,NaN,NaN,NaN,NaN
1,1,2020,NaN,NaN,NaN,NaN
101,21,2020,NaN,NaN,NaN,NaN
299,60,2023,NaN,NaN,NaN,NaN
349,70,2023,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...
7844,1569,2023,NaN,NaN,NaN,NaN
7970,1595,2019,NaN,NaN,NaN,NaN
7971,1595,2020,NaN,NaN,NaN,NaN
7972,1595,2021,NaN,NaN,NaN,NaN


# Export Data

In [40]:
!pip install --upgrade -q gspread gspread-dataframe

import gspread
from gspread_dataframe import set_with_dataframe
from google.colab import auth
from google.auth import default
import pandas as pd

# Bước 1: Xác thực Google Account
auth.authenticate_user()

# Bước 2: Lấy credentials và kết nối Google Sheets
creds, _ = default()
gc = gspread.authorize(creds)

# Bước 3: Tạo file Google Sheets
spreadsheet = gc.create("Gcontest_2025_Cleaned_Data")

# Bước 4: Tạo từng worksheet với kích thước phù hợp với số dòng, cột thực tế
worksheet_bs = spreadsheet.add_worksheet(
    title="Balance Sheet",
    rows=str(len(balance_sheet_filled_all) + 10),
    cols=str(len(balance_sheet_filled_all.columns) + 5)
)

worksheet_is = spreadsheet.add_worksheet(
    title="Income Statement",
    rows=str(len(is_filled) + 10),
    cols=str(len(is_filled.columns) + 5)
)

worksheet_cf = spreadsheet.add_worksheet(
    title="Cash Flow",
    rows=str(len(cf_filled) + 10),
    cols=str(len(cf_filled.columns) + 5)
)

# Bước 5: Ghi dữ liệu
set_with_dataframe(worksheet_bs, balance_sheet_filled_all)
set_with_dataframe(worksheet_is, is_filled)
set_with_dataframe(worksheet_cf, cf_filled)

# Bước 6: Xoá Sheet1 mặc định
default_ws = spreadsheet.worksheet("Sheet1")
spreadsheet.del_worksheet(default_ws)

# Bước 7: In link Google Sheet
print("📎 Link file Google Sheets:", spreadsheet.url)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.1 MB/s eta 0:00:00
📎 Link file Google Sheets: https://docs.google.com/spreadsheets/d/16ohnRg-9dOi6dr9IHwhgp_fxgsVKPdFhgYsdXQ6CA_c
